In [0]:
%sql


SELECT
    'Bronze Commits' AS dataset,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT record_key) AS unique_rows
FROM github_health.bronze.commits

UNION ALL

SELECT 'Bronze Issues',
       COUNT(*), COUNT(DISTINCT record_key)
FROM github_health.bronze.issues

UNION ALL

SELECT 'Bronze Pulls',
       COUNT(*), COUNT(DISTINCT record_key)
FROM github_health.bronze.pulls

UNION ALL

SELECT 'Silver Commits',
       COUNT(*), COUNT(DISTINCT record_key)
FROM github_health.silver.commits

UNION ALL

SELECT 'Silver Issues',
       COUNT(*), COUNT(DISTINCT record_key)
FROM github_health.silver.issues

UNION ALL

SELECT 'Silver Pulls',
       COUNT(*), COUNT(DISTINCT record_key)
FROM github_health.silver.pulls;


In [0]:
%sql

SELECT
    'Bronze' AS layer,
    COUNT(*) AS quarantined_rows
FROM github_health.ops.bronze_quarantine

UNION ALL

SELECT
    'Silver',
    COUNT(*)
FROM github_health.ops.silver_quarantine;


In [0]:
%sql

SELECT
    notebook_name,
    layer,
    status,
    rows_processed,
    upserted_rows,
    quarantined_rows,
    start_time
FROM github_health.ops.pipeline_execution_logs
WHERE batch_id = '92199a9d-c132-4525-92c9-5094e8e1f310'
ORDER BY start_time DESC
LIMIT 25;


In [0]:

import json

result = dbutils.notebook.run(
    "./raw_to_bronze",
    0,
    {
        "batch_id": "92199a9d-c132-4525-92c9-5094e8e1f310",
        "run_date": "2026-10-08",
        "load_type": "FULL",
        "repository": "apache/spark",
        "entity_type": "issues",
        "file_path": next(
            x["path"]
            for x in json.loads(
                open(
                    "/Volumes/github_health/raw/bronze_data/"
                    "bronze/_manifests/"
                    "full_load_2026-10-08_"
                    "92199a9d-c132-4525-92c9-5094e8e1f310.json"
                ).read()
            )["files"]
            if x["repository"] == "apache/spark"
            and x["entity"] == "issues"
        )
    }
)

print(result)


In [0]:

import json
from pyspark.sql import functions as F

batch_id = "92199a9d-c132-4525-92c9-5094e8e1f310"

def silver_snapshot():
    rows = (
        spark.table("github_health.silver.issues")
        .filter(F.col("repository") == "apache/spark")
        .select("record_key", "load_timestamp")
        .collect()
    )
    return {
        r["record_key"]: r["load_timestamp"]
        for r in rows
    }

# 1. Save existing Silver timestamps
before = silver_snapshot()

# 2. Rerun identical data
result = json.loads(dbutils.notebook.run(
    "./bronze_to_silver",
    0,
    {
        "batch_id": batch_id,
        "run_date": "2026-10-08",
        "load_type": "FULL",
        "repository": "apache/spark",
        "entity_type": "issues"
    }
))

# 3. Check timestamps again
after = silver_snapshot()

print("Silver result:", result)
print("Records before:", len(before))
print("Records after:", len(after))
print("Timestamps unchanged:", before == after)

assert result["status"] == "SUCCESS"
assert result["upserted"] == 0
assert before == after

print("SILVER IDEMPOTENCY TEST PASSED")


In [0]:
%sql

SELECT repository, entity, layer,
       last_successful_watermark
FROM github_health.ops.pipeline_watermarks
WHERE layer = 'RAW'
ORDER BY repository, entity;




In [0]:
%sql

SELECT
    repository,
    entity,
    batch_id,
    last_successful_watermark
FROM github_health.ops.pipeline_watermarks
WHERE layer = 'RAW'
ORDER BY repository, entity;
